### Part 1
Fetch RSS Feed into the context

In [26]:
import requests

In [27]:
ADVERTS_RSS_URL = "https://www.adverts.ie/for-sale/rss/$"

In [28]:
def fetch_rss_feed(url: str, timeout: int = 15) -> str:
  """Fetch raw XML/RSS content with a descriptive User-Agent and timeout."""
  headers = {
      "User-Agent": (
          "AcademicDataPipeline/1.0 (Student AI Project; contact via 20101674@mydbs.ie)"
      ),
      "Accept": "application/rss+xml, application/xml, text/xml;q=0.9, */*;q=0.8",
  }

  response = requests.get(url, headers=headers, timeout=timeout)
  response.raise_for_status()  # Raises HTTPError on 4xx/5xx responses
  return response.text

In [29]:
raw_xml = fetch_rss_feed(ADVERTS_RSS_URL)
raw_xml

'<?xml version="1.0" encoding="utf-8"?>\n\n<rss version="2.0">\n\t<channel>\n\t<title>Adverts.ie - Recent Ads All 965,585 listings</title>\n\t<link>https://www.adverts.ie/for-sale/</link>\n\t<description>Adverts.ie - Recent Ads All 965,585 listings</description>\n\t<language>en</language>\n\n<item>\n\t<title>Horrid Henry Books by Francesca Simon</title>\n\t\t<price><![CDATA[5]]></price>\n\t\t\t<county><![CDATA[Cavan]]></county>\n\t\t\t<area><![CDATA[Cootehill]]></area>\n\t\t\t<image_url>https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiMjk1MDNmZDg3MmRlNTJmYzc3MGRiOWY4MzdkZGUwOTA5MWQ2NDA3MmE0NzM4NWM1M2MwMzFhZThkN2MwOGNhMC5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=e3b670e3d8c118dd742e551367f2a493256d8b07a4ebc3faf3fba161f050cc79</image_url>\n\t\t<link>https://www.adverts.ie/children-babies/horrid-henry-books-by-francesca-simon/29628326</link>\t\n\t<guid>29628326</guid>\n\t<pubDate>Tue, 24 Nov 2026 22:

### Part 2
Store the raw RSS feed into designated file and then parse the XML Response.

Folder structure for storing feed: YEAR -> MONTH -> DAY -> NOW.xml (One file per xml response is stored in day, month and year wise manner)

In [30]:
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
import xml.etree.ElementTree as ET

In [31]:
RAW_DIR = Path("data/raw")

In [32]:
def save_raw_xml(xml_bytes: bytes, base_dir: Path = RAW_DIR) -> Path:
    """Save the response as data/raw/<year>/<month>/<day>/<timestamp>.xml"""
    now = datetime.now(timezone.utc)
    folder = base_dir / f"{now:%Y}" / f"{now:%m}" / f"{now:%d}"
    folder.mkdir(parents=True, exist_ok=True)

    file_path = folder / f"{now:%Y-%m-%d_%H%M%S}.xml"
    file_path.write_bytes(xml_bytes)
    return file_path


In [33]:
def get_text(item, tag):
    """Return the text inside a tag, or None if it is missing or empty."""
    element = item.find(tag)
    if element is not None and element.text:
        return element.text.strip()
    return None


def parse_date(value):
    """Convert an RSS date string into a datetime."""
    if value is None:
        return None
    return parsedate_to_datetime(value)


def parse_rss(xml_bytes: bytes) -> list[dict]:
    """Turn the RSS XML into a list of listing dictionaries."""
    root = ET.fromstring(xml_bytes)
    listings = []

    for item in root.iter("item"):
        listings.append({
            "guid": get_text(item, "guid"),
            "title": get_text(item, "title"),
            "link": get_text(item, "link"),
            "description": get_text(item, "description"),
            "category": get_text(item, "category"),
            "published_at": parse_date(get_text(item, "pubDate")),
        })
    return listings

In [34]:
saved_path = save_raw_xml(raw_xml.encode("utf-8")) # converts into bytes
print(saved_path, len(listings))

data\raw\2026\11\24\2026-11-24_224403.xml 30


In [35]:
listings = parse_rss(raw_xml)
listings

[{'guid': '29628326',
  'title': 'Horrid Henry Books by Francesca Simon',
  'link': 'https://www.adverts.ie/children-babies/horrid-henry-books-by-francesca-simon/29628326',
  'description': '<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a href="https://www.adverts.ie/children-babies/horrid-henry-books-by-francesca-simon/29628326">\n\t\t\t\t\t\t\t<img src="https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiMjk1MDNmZDg3MmRlNTJmYzc3MGRiOWY4MzdkZGUwOTA5MWQ2NDA3MmE0NzM4NWM1M2MwMzFhZThkN2MwOGNhMC5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=e3b670e3d8c118dd742e551367f2a493256d8b07a4ebc3faf3fba161f050cc79" alt="Horrid Henry Books by Francesca Simon" width="180" height="120" />\n\t\t\t\t\t\t</a>\n\t\t\t\t\t\t\t\t</div>\n\t\t\t\t<div>\n\t\t\t\t\t<div>\t\t\t\t\n\t\t\t\t\t\t<h4>\n\t\t\t\t\t\t\t<a href="https://www.adverts.ie/children-babies/horrid-henry-books-by-francesca-simon/29628326" title="Horrid Henry Books by